# Chapter 10: Conics and Perspectivity

**Source orientation.** Sections 10.1-10.7, printed pages 167-188, PDF pages 189-210. The source span was used for structure, terminology, and theorem coverage only; the prose, code, diagrams, and checks below are original.

**Chapter goal.** Build a computational picture of why a nondegenerate conic behaves like a projective line: five points determine it, a cross-ratio can be read from any point on it, projectively paired pencils generate it, transformations preserving it restrict to PGL(2), and Hesse's transfer principle turns incidence with conics into one-dimensional pair relations.

The notebook is organized around seven inspectable objects: a five-point conic matrix and its bracket-pencil construction; the same conic seen from several viewpoints with equal cross-ratio; a sweep of corresponding lines from two bundles; a PGL(2) action lifted to a projective transformation of the conic; Hesse pairs on a parabola; Pascal and Brianchon duality; and a harmonic-point sanity check.


## Computational Translation Guide

| Source idea | Computational object in this notebook | Library route | What to inspect |
| --- | --- | --- | --- |
| A conic through five points | A null vector of the 5 by 6 quadratic design matrix, and the equivalent bracket pencil from two degenerate conics | NumPy for SVD and homogeneous arithmetic; Matplotlib for the pencil diagram | The five marked points have near-zero quadratic residuals, and the SVD and bracket-pencil matrices differ only by scale. |
| Cross-ratio on a conic | The determinant formula `(1,2;3,4)_p` evaluated from several points `p` on the same conic | NumPy determinants; Matplotlib plot of viewpoints and secants | The measured value is flat as the viewpoint moves, except at excluded coincidences. |
| Perspective generation | Corresponding lines from two bundles through fixed conic points; their meets form the locus | Plotly HTML for an inspectable sweep | The current meet lies on both lines and has conic residual near zero. |
| Transformations preserving a conic | A 2 by 2 Mobius matrix lifted by the symmetric-square representation to a 3 by 3 projective matrix | NumPy plus Plotly | The lifted matrix scales the conic form and preserves cross-ratio on parameters. |
| Hesse transfer | A line is represented by its two intersections with a parabola; concurrence becomes a quadrilateral-set equation | SymPy for the exact determinant identity; Matplotlib for the parabola device | Equal y-axis intercepts are equal products, and the quadset residual vanishes. |
| Pascal/Brianchon duality | Opposite-side meets of an inscribed hexagon, and opposite-vertex joins of a tangent hexagon | NumPy joins/meets; Matplotlib and NetworkX | Pascal points are collinear; Brianchon lines are concurrent; the proof graph shows the transfer dependency. |

Homogeneous points are triples `(x,y,z)`, lines are triples `(a,b,c)` with `a x + b y + c z = 0`, `join` and `meet` are cross products, and the bracket `[a,b,c]` is a 3 by 3 determinant. A conic is a symmetric matrix `A` evaluated by `p.T @ A @ p`.


In [ ]:
from pathlib import Path
import sys

START = Path.cwd().resolve()
BOOK_ROOT = None
for candidate in [START, *START.parents]:
    if (candidate / "AGENTS.md").exists() and (candidate / "Perspectives on Projective Geometry.pdf").exists():
        BOOK_ROOT = candidate
        break
    nested = candidate / "Perspectives-on-Projective-Geometry"
    if (nested / "AGENTS.md").exists() and (nested / "Perspectives on Projective Geometry.pdf").exists():
        BOOK_ROOT = nested
        break
if BOOK_ROOT is None:
    raise RuntimeError("Could not discover the Perspectives on Projective Geometry course root")
if str(BOOK_ROOT) not in sys.path:
    sys.path.insert(0, str(BOOK_ROOT))

ARTIFACT_ROOT = BOOK_ROOT / "artifacts" / "chapter-10-conics-and-perspectivity"
ARTIFACT_ROOT


In [ ]:
import math

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch
import networkx as nx
import numpy as np
import plotly.graph_objects as go
import sympy as sp

from utils.artifacts import (
    assert_artifacts,
    book_relative,
    display_artifact,
    ensure_artifact_root,
    image_stats,
    save_json,
    save_table,
)

ensure_artifact_root(ARTIFACT_ROOT)
FIG_DIR = ARTIFACT_ROOT / "figures"
HTML_DIR = ARTIFACT_ROOT / "html"
CHECK_DIR = ARTIFACT_ROOT / "checks"

ARTIFACTS = []
TABLES = []
CHECKS = {}

plt.ioff()


def register_artifact(path):
    path = Path(path)
    if path not in ARTIFACTS:
        ARTIFACTS.append(path)
    return path


def register_table(path):
    path = Path(path)
    if path not in TABLES:
        TABLES.append(path)
    return path


def save_fig(fig, filename):
    path = FIG_DIR / filename
    fig.savefig(path, dpi=170, bbox_inches="tight", facecolor="white")
    plt.close(fig)
    return register_artifact(path)


def write_plotly(fig, filename):
    path = HTML_DIR / filename
    fig.write_html(path, include_plotlyjs="inline", full_html=True)
    return register_artifact(path)


In [ ]:
EPS = 1e-10


def hpoint(x, y, z=1.0):
    return np.array([float(x), float(y), float(z)], dtype=float)


def affine(p, tol=1e-12):
    p = np.asarray(p, dtype=float)
    if abs(p[2]) < tol:
        raise ValueError("point at infinity")
    return p[:2] / p[2]


def join(p, q):
    return np.cross(np.asarray(p, dtype=float), np.asarray(q, dtype=float))


def meet(l, m):
    return np.cross(np.asarray(l, dtype=float), np.asarray(m, dtype=float))


def bracket(a, b, c):
    return float(np.linalg.det(np.column_stack([a, b, c])))


def cross_ratio(a, b, c, d):
    return ((a - c) * (b - d)) / ((a - d) * (b - c))


def circle_point(t):
    return hpoint(np.cos(t), np.sin(t), 1.0)


def circle_tangent(p):
    p = np.asarray(p, dtype=float)
    return np.array([p[0], p[1], -p[2]], dtype=float)


def parabola_point(t):
    return hpoint(t, t * t, 1.0)


def parabola_tangent(t):
    return np.array([-2.0 * t, 1.0, t * t], dtype=float)


def conic_matrix_from_coeff(coeff):
    a, b, c, d, e, f = np.asarray(coeff, dtype=float)
    return np.array(
        [[a, d / 2.0, e / 2.0], [d / 2.0, b, f / 2.0], [e / 2.0, f / 2.0, c]],
        dtype=float,
    )


def fit_conic_matrix(points):
    rows = []
    for x, y, z in points:
        rows.append([x * x, y * y, z * z, x * y, x * z, y * z])
    design = np.asarray(rows, dtype=float)
    _, singular_values, vt = np.linalg.svd(design)
    coeff = vt[-1]
    return conic_matrix_from_coeff(coeff), coeff, singular_values, design


def pair_conic_matrix(line_a, line_b):
    m = np.outer(line_a, line_b)
    return (m + m.T) / 2.0


def conic_from_bracket_pencil(p1, p2, p3, p4, q):
    g1 = join(p1, p3)
    g2 = join(p2, p4)
    h1 = join(p1, p4)
    h2 = join(p2, p3)
    g = np.outer(g1, g2)
    h = np.outer(h1, h2)
    m = (q @ h @ q) * g - (q @ g @ q) * h
    return (m + m.T) / 2.0


def conic_value(A, p):
    p = np.asarray(p, dtype=float)
    return float(p @ A @ p)


def conic_values_on_grid(A, xlim, ylim, n=420):
    xs = np.linspace(xlim[0], xlim[1], n)
    ys = np.linspace(ylim[0], ylim[1], n)
    X, Y = np.meshgrid(xs, ys)
    Z = (
        A[0, 0] * X * X
        + A[1, 1] * Y * Y
        + A[2, 2]
        + 2 * A[0, 1] * X * Y
        + 2 * A[0, 2] * X
        + 2 * A[1, 2] * Y
    )
    return X, Y, Z


def plot_line_on_box(ax, line, xlim, ylim, **kwargs):
    a, b, c = np.asarray(line, dtype=float)
    pts = []
    if abs(b) > EPS:
        for x in xlim:
            y = -(a * x + c) / b
            if ylim[0] - 1e-9 <= y <= ylim[1] + 1e-9:
                pts.append((x, y))
    if abs(a) > EPS:
        for y in ylim:
            x = -(b * y + c) / a
            if xlim[0] - 1e-9 <= x <= xlim[1] + 1e-9:
                pts.append((x, y))
    unique = []
    for pt in pts:
        if not any(np.linalg.norm(np.asarray(pt) - np.asarray(other)) < 1e-7 for other in unique):
            unique.append(pt)
    if len(unique) >= 2:
        best = (unique[0], unique[1], -1.0)
        for i in range(len(unique)):
            for j in range(i + 1, len(unique)):
                dist = np.linalg.norm(np.asarray(unique[i]) - np.asarray(unique[j]))
                if dist > best[2]:
                    best = (unique[i], unique[j], dist)
        ax.plot([best[0][0], best[1][0]], [best[0][1], best[1][1]], **kwargs)


def cr_seen_from(view, p1, p2, p3, p4):
    return (
        bracket(view, p1, p3)
        * bracket(view, p2, p4)
        / (bracket(view, p1, p4) * bracket(view, p2, p3))
    )


def frobenius_scale_residual(A, B):
    scale = float(np.sum(A * B) / np.sum(B * B))
    return float(np.linalg.norm(A - scale * B)), scale


def max_abs(values):
    return float(np.max(np.abs(np.asarray(values, dtype=float))))


## Visualization Planner Pass

The route below is the source-specific storyboard implemented in this notebook. Each visual has an inspection target and a check; the saved `storyboard.json` records the same plan for later audit.


In [ ]:
storyboard = {
    "chapter goal": "Explain conics as projective-line objects controlled by cross-ratio, perspectivity, and Hesse transfer.",
    "source span read": "Perspectives on Projective Geometry, Chapter 10, sections 10.1-10.7; printed pp. 167-188; PDF pp. 189-210.",
    "concept inventory": [
        "five-point conic fitting by a 5 by 6 homogeneous linear system",
        "degenerate line-pair conics and the bracket pencil through four base points",
        "cross-ratio of four conic points seen from a fifth conic point",
        "projective line-bundle correspondence whose meets sweep a conic",
        "PGL(2) action on a conic via parameter cross-ratio",
        "Hesse line-to-pair transfer, including tangent/double and complex-pair caveats",
        "Matiyasevich parabola multiplication as a real model for Hesse concurrence",
        "Pascal collinearity and Brianchon concurrence as dual conic incidence theorems",
        "harmonic point construction from two tangents and a secant",
    ],
    "library routing table": [
        {"concept": "five-point conic and bracket pencil", "representation": "2D conic contours plus line pairs", "library": "NumPy, Matplotlib", "why": "linear algebra residuals and durable labeled incidence diagrams", "fallback": "SymPy exact examples for small integer data"},
        {"concept": "perspective generation sweep", "representation": "interactive line-bundle sweep", "library": "Plotly", "why": "the moving line pair and meet are easier to inspect with a slider", "fallback": "static multiple-line sweep PNG"},
        {"concept": "Hesse transfer and Pascal proof", "representation": "parabola multiplication, proof dependency graph", "library": "SymPy, Matplotlib, NetworkX", "why": "exact identity plus dependency structure expose the proof move", "fallback": "CSV residual table"},
    ],
    "visual sequence": [
        {"concept": "chapter route through conic-as-line", "artifact filename": "figures/conic-perspectivity-route-map.png", "learner inspection target": "follow which invariant carries each section", "validation/invariant": "storyboard and route nodes agree"},
        {"concept": "five-point conic fitting", "artifact filename": "figures/five-point-conic-pencil.png", "learner inspection target": "compare SVD conic with bracket-pencil conic", "validation/invariant": "five marked residuals and matrix scale residual are near zero"},
        {"concept": "conic cross-ratio viewpoints", "artifact filename": "figures/conic-cross-ratio-viewpoints.png", "learner inspection target": "read equal values from different viewpoints", "validation/invariant": "max cross-ratio error across viewpoints"},
        {"concept": "perspective generation sweep", "artifact filename": "html/perspective-generation-sweep.html", "learner inspection target": "move the corresponding line pair and watch the meet stay on the conic", "validation/invariant": "meet lies on both lines and on the conic"},
        {"concept": "projective transformations preserving conic", "artifact filename": "html/projective-transform-conic-action.html", "learner inspection target": "compare old and Mobius-moved parameters", "validation/invariant": "T.T Q T is a scalar multiple of Q"},
        {"concept": "Hesse transfer checks", "artifact filename": "figures/hesse-transfer-parabola-multiplication.png", "learner inspection target": "see equal products as equal y-axis intercepts", "validation/invariant": "quadset residual and SymPy determinant vanish"},
        {"concept": "Pascal and Brianchon duality", "artifact filename": "figures/pascal-brianchon-duality.png", "learner inspection target": "compare collinearity of opposite-side meets with concurrence of opposite-vertex joins", "validation/invariant": "Pascal incidence and Brianchon incidence residuals"},
        {"concept": "Hesse-to-Pascal proof graph", "artifact filename": "figures/hesse-pascal-transfer-proof-graph.png", "learner inspection target": "trace the transfer principle from pairs to Pascal", "validation/invariant": "directed graph is acyclic"},
    ],
    "artifact plan": "All artifacts are written under artifacts/chapter-10-conics-and-perspectivity/{figures,html,tables,checks}.",
    "computational checks": ["five-point residuals", "SVD versus bracket-pencil matrix scale residual", "conic cross-ratio viewpoint residual", "perspective sweep incidence and conic residuals", "symmetric-square conic-form preservation", "Hesse determinant and quadrilateral-set residual", "Pascal collinearity and Brianchon concurrence residuals", "harmonic cross-ratio residual", "artifact existence and nonzero size"],
    "proof-visualization strategy": "Use a NetworkX dependency graph plus determinant residuals: Hesse pair transfer proves concurrence as a quadset relation; multiplying two quadset relations gives Pascal; Brianchon is the dual statement.",
    "implementation notes": "The notebook keeps all chapter-specific construction code inline and writes only the assigned notebook and chapter artifact subtree.",
    "risks": ["near-degenerate sample parameters can make intersections numerically large", "HTML interactivity depends on notebook/browser support but static checks are stored in JSON"],
    "acceptance criteria": ["direct nbclient execution succeeds", "no generic scaffold renderer remains", "visual-checks.json reports all_files_exist and cross_ratio_error below 1e-9", "scoped diff check is clean"],
}

save_json(storyboard, ARTIFACT_ROOT, "checks", "storyboard.json")

route_labels = [
    ("Five points", "nullspace + bracket pencil"),
    ("Cross-ratio", "same value from any conic point"),
    ("Line bundles", "paired lines sweep the locus"),
    ("PGL(2) on C", "Mobius action lifted to RP^2"),
    ("Hesse transfer", "lines become point pairs"),
    ("Pascal / Brianchon", "quadsets and duality"),
]
fig, ax = plt.subplots(figsize=(12.5, 3.6))
ax.set_axis_off()
colors = ["#4c78a8", "#f58518", "#54a24b", "#b279a2", "#e45756", "#72b7b2"]
for i, (title, subtitle) in enumerate(route_labels):
    x = i * 1.9
    box = FancyBboxPatch((x, 0.35), 1.55, 0.95, boxstyle="round,pad=0.04,rounding_size=0.03", linewidth=1.4, edgecolor=colors[i], facecolor="#ffffff")
    ax.add_patch(box)
    ax.text(x + 0.775, 1.02, title, ha="center", va="center", fontsize=10.5, weight="bold", color="#1f2933")
    ax.text(x + 0.775, 0.68, subtitle, ha="center", va="center", fontsize=8.5, color="#344054")
    if i < len(route_labels) - 1:
        ax.annotate("", xy=(x + 1.82, 0.83), xytext=(x + 1.58, 0.83), arrowprops={"arrowstyle": "->", "lw": 1.6, "color": "#475467"})
ax.text(0, 1.62, "Chapter 10 route: every step keeps a projective invariant visible", fontsize=12.5, weight="bold", color="#111827")
ax.set_xlim(-0.12, 11.2)
ax.set_ylim(0.2, 1.9)
route_path = save_fig(fig, "conic-perspectivity-route-map.png")
display_artifact(route_path, width=900)


## 10.1 Five Points, Two Computations

A quadratic conic has six coefficients, but simultaneous scaling does not change the curve. Five point conditions normally leave a one-dimensional nullspace. The chapter also gives a structural computation: use four base points to build two degenerate conics, each a pair of lines, then choose the linear combination that also passes through the fifth point.

The diagram below uses both computations on the same five points. The left panel shows the fitted conic and the two degenerate line-pair ingredients. The right panel shows nearby members of the four-point pencil, so the fifth point's role is visible.


In [ ]:
five_params = np.array([-2.45, -1.05, 0.25, 1.35, 2.35])
five_points = [circle_point(t) for t in five_params]
A_svd, conic_coeff, singular_values, design_matrix = fit_conic_matrix(five_points)
A_pencil = conic_from_bracket_pencil(*five_points[:4], five_points[4])
fit_matrix_residual, fit_matrix_scale = frobenius_scale_residual(A_svd, A_pencil)
five_point_residuals = [conic_value(A_svd, p) for p in five_points]
pencil_point_residuals = [conic_value(A_pencil, p) for p in five_points]

G13_24 = pair_conic_matrix(join(five_points[0], five_points[2]), join(five_points[1], five_points[3]))
G14_23 = pair_conic_matrix(join(five_points[0], five_points[3]), join(five_points[1], five_points[2]))

xlim, ylim = (-1.55, 1.55), (-1.55, 1.55)
fig, axes = plt.subplots(1, 2, figsize=(12, 5.2), sharex=True, sharey=True)
for ax in axes:
    ax.set_aspect("equal")
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    ax.axhline(0, color="#d0d5dd", lw=0.8)
    ax.axvline(0, color="#d0d5dd", lw=0.8)

X, Y, Z = conic_values_on_grid(A_svd, xlim, ylim)
axes[0].contour(X, Y, Z, levels=[0], colors=["#1f77b4"], linewidths=2.4)
for line, color in [
    (join(five_points[0], five_points[2]), "#f58518"),
    (join(five_points[1], five_points[3]), "#f58518"),
    (join(five_points[0], five_points[3]), "#54a24b"),
    (join(five_points[1], five_points[2]), "#54a24b"),
]:
    plot_line_on_box(axes[0], line, xlim, ylim, color=color, lw=1.25, alpha=0.72, linestyle="--")
for i, p in enumerate(five_points, start=1):
    xy = affine(p)
    axes[0].scatter(*xy, s=48, color="#111827", zorder=4)
    axes[0].text(xy[0] + 0.045, xy[1] + 0.045, "q" if i == 5 else str(i), fontsize=10, weight="bold")
axes[0].set_title("One conic selected by five point conditions")
axes[0].text(-1.45, -1.39, f"max |p^T A p| = {max_abs(five_point_residuals):.1e}", fontsize=9, color="#344054")

for scale, color in zip([-1.8, -0.65, 0.15, 0.8, 1.7], ["#a6cee3", "#1f78b4", "#33a02c", "#fb9a99", "#e31a1c"]):
    M = G13_24 + scale * G14_23
    X, Y, Z = conic_values_on_grid(M, xlim, ylim)
    axes[1].contour(X, Y, Z, levels=[0], colors=[color], linewidths=1.6, alpha=0.9)
for i, p in enumerate(five_points[:4], start=1):
    xy = affine(p)
    axes[1].scatter(*xy, s=45, color="#111827", zorder=4)
    axes[1].text(xy[0] + 0.045, xy[1] + 0.045, str(i), fontsize=10, weight="bold")
qxy = affine(five_points[4])
axes[1].scatter(*qxy, s=62, color="#e45756", zorder=4)
axes[1].text(qxy[0] + 0.045, qxy[1] + 0.045, "q", fontsize=10, weight="bold", color="#b42318")
axes[1].set_title("Four-point pencil; q picks a member")
axes[1].text(-1.45, -1.39, f"SVD vs bracket matrix residual = {fit_matrix_residual:.1e}", fontsize=9, color="#344054")

for ax in axes:
    ax.set_xlabel("x/z")
axes[0].set_ylabel("y/z")
fig.suptitle("Five-point conic fitting and the bracket-pencil construction", fontsize=14, weight="bold")
five_point_path = save_fig(fig, "five-point-conic-pencil.png")
CHECKS.update({
    "five_point_max_residual": max_abs(five_point_residuals),
    "pencil_max_residual": max_abs(pencil_point_residuals),
    "svd_pencil_matrix_residual": fit_matrix_residual,
    "conic_design_rank": int(np.linalg.matrix_rank(design_matrix)),
})
display_artifact(five_point_path, width=880)
CHECKS


The two residuals test different parts of the chapter's claim. The point residual confirms that the quadratic matrix vanishes at the five data points. The matrix residual confirms that the determinant/bracket construction and the linear-algebra nullspace describe the same projective conic, up to scale.


## 10.2 Cross-Ratio Seen From the Conic

For four fixed conic points, the expression `[p,1,3][p,2,4] / ([p,1,4][p,2,3])` is the cross-ratio of the four secants through the viewpoint `p`. The theorem is that this value does not depend on which fifth point `p` on the same conic is used, as long as the determinants are defined.


In [ ]:
base_points = five_points[:4]
view_params = np.array([-2.9, -2.05, -0.45, 0.85, 1.85, 2.8])
view_points = [circle_point(t) for t in view_params]
cr_values = [cr_seen_from(v, *base_points) for v in view_points]
cr_reference = cr_values[0]
cr_errors = [abs(v - cr_reference) for v in cr_values]

rows = [
    {"view_parameter": float(t), "cross_ratio_seen_from_view": float(v), "absolute_error_from_first_view": float(e)}
    for t, v, e in zip(view_params, cr_values, cr_errors)
]
cr_table = register_table(save_table(rows, ARTIFACT_ROOT, "tables", "conic-cross-ratio-viewpoint-values.csv"))

fig, axes = plt.subplots(1, 2, figsize=(12.5, 5.2))
curve_t = np.linspace(-math.pi, math.pi, 500)
curve = np.array([affine(circle_point(t)) for t in curve_t])
axes[0].set_aspect("equal")
axes[0].plot(curve[:, 0], curve[:, 1], color="#1f77b4", lw=2.2)
for i, p in enumerate(base_points, start=1):
    xy = affine(p)
    axes[0].scatter(*xy, s=52, color="#111827", zorder=5)
    axes[0].text(xy[0] + 0.04, xy[1] + 0.04, str(i), fontsize=10, weight="bold")
for view, color, label in [(view_points[1], "#e45756", "view p"), (view_points[3], "#54a24b", "view q")]:
    vxy = affine(view)
    axes[0].scatter(*vxy, s=70, color=color, zorder=6)
    axes[0].text(vxy[0] + 0.05, vxy[1] + 0.05, label, fontsize=9, weight="bold", color=color)
    for target in base_points:
        plot_line_on_box(axes[0], join(view, target), (-1.35, 1.35), (-1.35, 1.35), color=color, lw=0.9, alpha=0.42)
axes[0].set_xlim(-1.35, 1.35)
axes[0].set_ylim(-1.35, 1.35)
axes[0].set_title("Two viewpoints read the same four-point value")
axes[0].set_xlabel("x/z")
axes[0].set_ylabel("y/z")

axes[1].plot(view_params, cr_values, marker="o", color="#7a5195", lw=2)
axes[1].axhline(cr_reference, color="#344054", lw=1, linestyle="--")
axes[1].set_title("Cross-ratio is constant as the viewpoint moves")
axes[1].set_xlabel("view parameter on the conic")
axes[1].set_ylabel("determinant cross-ratio")
axes[1].grid(True, alpha=0.25)
axes[1].text(view_params[0], cr_reference + 0.02, f"max error {max(cr_errors):.1e}", fontsize=9, color="#344054")

fig.suptitle("Conic cross-ratio: four fixed points, moving viewpoint", fontsize=14, weight="bold")
cr_path = save_fig(fig, "conic-cross-ratio-viewpoints.png")
CHECKS.update({"cross_ratio_error": float(max(cr_errors)), "cross_ratio_reference": float(cr_reference)})
display_artifact(cr_path, width=880)
print(book_relative(cr_table))


The right panel is the invariant tracker. A projective drawing can make the secant pencils look very different, but the determinant ratio is unchanged. This is the computational version of treating the conic itself as a model of the real projective line.


## 10.3 Perspective Generation Sweep

Reverse the previous theorem: choose two centers `p` and `q` and pair the line through `p` with a corresponding line through `q`. If the correspondence between the two line bundles is projective and not a perspectivity, the moving intersections trace a conic. Here the correspondence is induced by a parameter on the same circle, so every frame has a direct residual check.


In [ ]:
sweep_center_p = circle_point(-2.25)
sweep_center_q = circle_point(0.9)
sweep_ts = np.linspace(-2.95, 2.95, 90)
sweep_ts = np.array([t for t in sweep_ts if abs(t + 2.25) > 0.045 and abs(t - 0.9) > 0.045])
A_circle = np.diag([1.0, 1.0, -1.0])

locus_points = []
line_incidence_errors = []
conic_residuals = []
for t in sweep_ts:
    target = circle_point(t)
    line_p = join(sweep_center_p, target)
    line_q = join(sweep_center_q, target)
    intersection = meet(line_p, line_q)
    intersection = intersection / intersection[2]
    locus_points.append(intersection)
    line_incidence_errors.extend([abs(float(line_p @ intersection)), abs(float(line_q @ intersection))])
    conic_residuals.append(abs(conic_value(A_circle, intersection)))
locus_xy = np.array([affine(p) for p in locus_points])
curve_xy = np.array([affine(circle_point(t)) for t in np.linspace(-math.pi, math.pi, 500)])

fig = go.Figure()
fig.add_trace(go.Scatter(x=curve_xy[:, 0], y=curve_xy[:, 1], mode="lines", name="conic C", line={"color": "#1f77b4", "width": 3}))
centers = np.array([affine(sweep_center_p), affine(sweep_center_q)])
fig.add_trace(go.Scatter(x=centers[:, 0], y=centers[:, 1], mode="markers+text", text=["p", "q"], textposition="top center", name="bundle centers", marker={"size": 11, "color": "#111827"}))
fig.add_trace(go.Scatter(x=locus_xy[:, 0], y=locus_xy[:, 1], mode="markers", name="all computed meets", marker={"size": 5, "color": "rgba(228,87,86,0.45)"}))

frame_indices = np.linspace(0, len(sweep_ts) - 1, 32, dtype=int)
initial = frame_indices[0]

def segment_to_target(center, target):
    c = affine(center)
    a = affine(target)
    return [c[0], a[0]], [c[1], a[1]]

initial_target = circle_point(sweep_ts[initial])
px, py = segment_to_target(sweep_center_p, initial_target)
qx, qy = segment_to_target(sweep_center_q, initial_target)
fig.add_trace(go.Scatter(x=[locus_xy[initial, 0]], y=[locus_xy[initial, 1]], mode="markers", name="current meet", marker={"size": 13, "color": "#e45756"}))
fig.add_trace(go.Scatter(x=px, y=py, mode="lines", name="line from p", line={"color": "#f58518", "width": 2}))
fig.add_trace(go.Scatter(x=qx, y=qy, mode="lines", name="corresponding line from q", line={"color": "#54a24b", "width": 2}))

frames = []
for k in frame_indices:
    target = circle_point(sweep_ts[k])
    px, py = segment_to_target(sweep_center_p, target)
    qx, qy = segment_to_target(sweep_center_q, target)
    frames.append(go.Frame(
        name=f"t={sweep_ts[k]:.2f}",
        data=[
            go.Scatter(x=[locus_xy[k, 0]], y=[locus_xy[k, 1]], mode="markers", marker={"size": 13, "color": "#e45756"}),
            go.Scatter(x=px, y=py, mode="lines", line={"color": "#f58518", "width": 2}),
            go.Scatter(x=qx, y=qy, mode="lines", line={"color": "#54a24b", "width": 2}),
        ],
        traces=[3, 4, 5],
    ))
fig.frames = frames
fig.update_layout(
    title="Perspective generation: corresponding line bundles sweep a conic",
    width=820,
    height=620,
    xaxis={"scaleanchor": "y", "range": [-1.35, 1.35], "title": "x/z"},
    yaxis={"range": [-1.35, 1.35], "title": "y/z"},
    sliders=[{"steps": [{"args": [[frame.name], {"frame": {"duration": 0, "redraw": True}, "mode": "immediate"}], "label": frame.name, "method": "animate"} for frame in frames], "currentvalue": {"prefix": "parameter "}}],
    updatemenus=[{"type": "buttons", "buttons": [{"label": "play", "method": "animate", "args": [None, {"frame": {"duration": 180, "redraw": True}, "fromcurrent": True}]}, {"label": "pause", "method": "animate", "args": [[None], {"frame": {"duration": 0, "redraw": False}, "mode": "immediate"}]}]}],
)
sweep_path = write_plotly(fig, "perspective-generation-sweep.html")
CHECKS.update({
    "sweep_max_line_incidence_error": float(max(line_incidence_errors)),
    "sweep_max_conic_residual": float(max(conic_residuals)),
    "sweep_sample_count": int(len(sweep_ts)),
})
display_artifact(sweep_path, width="100%", height=620)


A perspectivity would collapse this story into a degenerate case: all corresponding intersections would lie on one fixed line. The HTML sweep makes the nondegenerate case visible because the meet moves around the conic instead of sliding along a line.


## 10.4 Projective Transformations Preserving a Conic

A nondegenerate conic is a projective line in disguise. Parameterize the parabola by the quadratic Veronese map `[u:v] -> [u^2 : u v : v^2]`. A 2 by 2 Mobius matrix acting on `[u:v]` lifts to a 3 by 3 matrix acting on `[u^2,uv,v^2]`. The conic form `x0*x2 - x1^2` is preserved up to the scalar `det(B)^2`.


In [ ]:
def veronese_point(t):
    return np.array([t * t, t, 1.0], dtype=float)


def mobius(t, B):
    a, b = B[0]
    c, d = B[1]
    return (a * t + b) / (c * t + d)


def symmetric_square(B):
    a, b = B[0]
    c, d = B[1]
    return np.array(
        [[a * a, 2 * a * b, b * b], [a * c, a * d + b * c, b * d], [c * c, 2 * c * d, d * d]],
        dtype=float,
    )

mobius_matrix = np.array([[1.12, -0.32], [0.18, 1.0]], dtype=float)
T_lift = symmetric_square(mobius_matrix)
Q_parabola = np.array([[0.0, 0.0, 0.5], [0.0, -1.0, 0.0], [0.5, 0.0, 0.0]], dtype=float)
form_preservation_residual = float(np.linalg.norm(T_lift.T @ Q_parabola @ T_lift - (np.linalg.det(mobius_matrix) ** 2) * Q_parabola))

param_samples = np.array([-1.9, -0.8, -0.15, 0.55, 1.35, 2.05])
image_params = np.array([mobius(t, mobius_matrix) for t in param_samples])
cr_before = cross_ratio(*param_samples[:4])
cr_after = cross_ratio(*image_params[:4])
transform_cr_error = abs(cr_before - cr_after)
transformed_residuals = [abs(veronese_point(t) @ Q_parabola @ veronese_point(t)) for t in image_params]

curve_t = np.linspace(-2.35, 2.35, 350)
curve_xy = np.array([affine(veronese_point(t)) for t in curve_t])
points_xy = np.array([affine(veronese_point(t)) for t in param_samples])
images_xy = np.array([affine(T_lift @ veronese_point(t)) for t in param_samples])

fig = go.Figure()
fig.add_trace(go.Scatter(x=curve_xy[:, 0], y=curve_xy[:, 1], mode="lines", name="conic x0*x2=x1^2", line={"color": "#1f77b4", "width": 3}))
fig.add_trace(go.Scatter(x=points_xy[:, 0], y=points_xy[:, 1], mode="markers+text", text=[f"t={t:.2g}" for t in param_samples], textposition="bottom center", name="original parameters", marker={"size": 9, "color": "#111827"}))
fig.add_trace(go.Scatter(x=images_xy[:, 0], y=images_xy[:, 1], mode="markers+text", text=[f"t'={t:.2g}" for t in image_params], textposition="top center", name="Mobius images", marker={"size": 10, "color": "#e45756", "symbol": "diamond"}))
for start, end in zip(points_xy, images_xy):
    fig.add_trace(go.Scatter(x=[start[0], end[0]], y=[start[1], end[1]], mode="lines", showlegend=False, line={"color": "rgba(84,162,75,0.55)", "width": 2}))
fig.update_layout(
    title="A Mobius map on RP^1 lifted to a projective transformation preserving the conic",
    width=820,
    height=560,
    xaxis={"title": "x0/x2", "range": [-0.2, 5.7]},
    yaxis={"title": "x1/x2", "range": [-2.4, 2.4], "scaleanchor": "x"},
)
transform_path = write_plotly(fig, "projective-transform-conic-action.html")
CHECKS.update({
    "projective_transform_form_residual": form_preservation_residual,
    "projective_transform_cross_ratio_error": float(transform_cr_error),
    "projective_transform_max_conic_residual": max_abs(transformed_residuals),
})
display_artifact(transform_path, width="100%", height=560)


This is the algebra behind the chapter's group statement. The conic-preserving transformations are not arbitrary 3 by 3 matrices; on the conic they behave exactly like projective transformations of a line.


## 10.5 Hesse Transfer: Lines Become Pairs

Hesse's transfer principle associates a line with the pair of points where it meets a fixed conic. On the real parabola `y=x^2`, the chord through parameters `a` and `b` hits the y-axis at `-a*b`. Thus two chords through the same point on the y-axis have equal products. That equality is the visible form of the quadrilateral-set relation used for concurrence.


In [ ]:
x_sym, y_sym = sp.symbols("x y")
mat_det = sp.Matrix([[-x_sym, y_sym, 0], [x_sym**2, y_sym**2, x_sym * y_sym], [1, 1, 1]]).det()
mat_det_simplified = sp.factor(mat_det)

pair_a, pair_b = -1.2, 1.5
pair_c = -0.75
pair_d = (pair_a * pair_b) / pair_c
intercept = -pair_a * pair_b

def rp1_point(t):
    return np.array([float(t), 1.0])


def rp1_infinity():
    return np.array([1.0, 0.0])


def bracket2(u, v):
    return float(np.linalg.det(np.column_stack([u, v])))

quad_lhs = bracket2(rp1_point(pair_a), rp1_infinity()) * bracket2(rp1_point(pair_c), rp1_point(pair_b)) * bracket2(rp1_point(0), rp1_point(pair_d))
quad_rhs = bracket2(rp1_point(pair_a), rp1_point(pair_d)) * bracket2(rp1_point(pair_c), rp1_infinity()) * bracket2(rp1_point(0), rp1_point(pair_b))
quadset_residual = abs(quad_lhs - quad_rhs)

hesse_rows = []
for name, u, v, role in [
    ("l", pair_a, pair_b, "first chord through the common point"),
    ("m", pair_c, pair_d, "second chord through the common point"),
    ("comparison-1", -0.9, (pair_a * pair_b) / -0.9, "same product, same y-axis intercept"),
    ("comparison-2", -1.5, (pair_a * pair_b) / -1.5, "same product, same y-axis intercept"),
]:
    hesse_rows.append({
        "line_pair": name,
        "first_parameter": u,
        "second_parameter": v,
        "product": u * v,
        "y_axis_intercept": -u * v,
        "shared_intercept_target": intercept,
        "absolute_intercept_error": abs((-u * v) - intercept),
        "inspection_role": role,
    })
hesse_table = register_table(save_table(hesse_rows, ARTIFACT_ROOT, "tables", "hesse-transfer-product-checks.csv"))

fig, ax = plt.subplots(figsize=(8.2, 6.2))
ax.set_aspect("equal")
tt = np.linspace(-2.4, 2.55, 500)
ax.plot(tt, tt * tt, color="#1f77b4", lw=2.5, label="parabola C")
ax.axvline(0, color="#344054", lw=1.3, label="y-axis")
ax.axhline(0, color="#d0d5dd", lw=0.8)
for (u, v, color, name) in [(pair_a, pair_b, "#e45756", "line l"), (pair_c, pair_d, "#54a24b", "line m")]:
    p_u, p_v = parabola_point(u), parabola_point(v)
    line = join(p_u, p_v)
    plot_line_on_box(ax, line, (-2.45, 2.6), (-1.2, 6.2), color=color, lw=2.0, label=name)
    for t, marker in [(u, "o"), (v, "s")]:
        ax.scatter(t, t * t, s=55, marker=marker, color=color, edgecolor="white", zorder=4)
        ax.plot([t, t], [0, t * t], color=color, lw=0.9, alpha=0.35)
        ax.scatter(t, 0, s=36, color=color, alpha=0.9)
        ax.text(t + 0.04, -0.22, f"{t:.2g}", fontsize=8, color=color)
ax.scatter(0, intercept, s=80, color="#111827", zorder=5)
ax.text(0.08, intercept + 0.1, f"common intercept = {intercept:.2f}", fontsize=9, weight="bold")
ax.set_xlim(-2.45, 2.6)
ax.set_ylim(-1.2, 6.2)
ax.set_xlabel("parameter axis / x-coordinate")
ax.set_ylabel("y")
ax.set_title("Hesse transfer on a parabola: concurrence becomes equal products", fontsize=13, weight="bold")
ax.legend(loc="upper left")
hesse_path = save_fig(fig, "hesse-transfer-parabola-multiplication.png")
CHECKS.update({
    "hesse_matiyasevich_symbolic_det_zero": bool(mat_det_simplified == 0),
    "hesse_quadset_residual": float(quadset_residual),
    "hesse_common_intercept_error": float(abs((-pair_a * pair_b) - (-pair_c * pair_d))),
})
display_artifact(hesse_path, width=700)
print("symbolic determinant:", mat_det_simplified)
print(book_relative(hesse_table))


The source chapter warns that a line may meet the conic in two real points, a tangent double point, or a conjugate complex pair. The diagram stays in the two-real-point case so the transfer is visible, but the algebraic residuals are the part that survives projective transformation and limiting cases.


## 10.6 Pascal and Brianchon as Transfer Checks

Pascal's theorem uses six points on a conic: the three intersections of opposite sides of the hexagon are collinear. Brianchon's theorem is the dual statement for six tangent lines: the joins of opposite tangent-hexagon vertices are concurrent.

The numerical check is direct. We compute every line and intersection with homogeneous joins/meets, then test one final incidence equation.


In [ ]:
pascal_params = np.array([-0.6380958342521121, -0.3575142650042924, -0.06583490930208313, 0.2054117401354092, 0.8907878458933132, 1.908456167959645])
P = [circle_point(t) for t in pascal_params]

pascal_X = meet(join(P[0], P[1]), join(P[3], P[4]))
pascal_Y = meet(join(P[1], P[2]), join(P[4], P[5]))
pascal_Z = meet(join(P[2], P[3]), join(P[5], P[0]))
pascal_line = join(pascal_X, pascal_Y)
pascal_residual = abs(float(pascal_line @ pascal_Z)) / np.linalg.norm(pascal_line)

T_lines = [circle_tangent(p) for p in P]
V = [meet(T_lines[i], T_lines[(i + 1) % 6]) for i in range(6)]
brianchon_L14 = join(V[0], V[3])
brianchon_L25 = join(V[1], V[4])
brianchon_L36 = join(V[2], V[5])
brianchon_point = meet(brianchon_L14, brianchon_L25)
brianchon_residual = abs(float(brianchon_L36 @ brianchon_point)) / np.linalg.norm(brianchon_L36)

fig, axes = plt.subplots(1, 2, figsize=(13, 5.8))
for ax in axes:
    ax.set_aspect("equal")
    ax.plot(curve[:, 0], curve[:, 1], color="#1f77b4", lw=2.2)
    ax.axhline(0, color="#e4e7ec", lw=0.7)
    ax.axvline(0, color="#e4e7ec", lw=0.7)

ax = axes[0]
hex_xy = np.array([affine(p) for p in P + [P[0]]])
ax.plot(hex_xy[:, 0], hex_xy[:, 1], color="#98a2b3", lw=1.2, alpha=0.8)
for i, p in enumerate(P, start=1):
    xy = affine(p)
    ax.scatter(*xy, s=48, color="#111827", zorder=4)
    ax.text(xy[0] + 0.035, xy[1] + 0.035, str(i), fontsize=9, weight="bold")
for line in [join(P[0], P[1]), join(P[3], P[4]), join(P[1], P[2]), join(P[4], P[5]), join(P[2], P[3]), join(P[5], P[0])]:
    plot_line_on_box(ax, line, (-0.4, 1.45), (-1.15, 1.2), color="#f58518", lw=0.9, alpha=0.45)
for label, point in [("X", pascal_X), ("Y", pascal_Y), ("Z", pascal_Z)]:
    xy = affine(point)
    ax.scatter(*xy, s=62, color="#e45756", zorder=5)
    ax.text(xy[0] + 0.025, xy[1] + 0.04, label, fontsize=10, weight="bold", color="#b42318")
plot_line_on_box(ax, pascal_line, (-0.4, 1.45), (-1.15, 1.2), color="#b42318", lw=2.0)
ax.set_xlim(-0.4, 1.45)
ax.set_ylim(-1.15, 1.2)
ax.set_title(f"Pascal: opposite-side meets collinear\nresidual {pascal_residual:.1e}")

ax = axes[1]
for line in T_lines:
    plot_line_on_box(ax, line, (-0.6, 3.0), (-1.25, 2.35), color="#54a24b", lw=1.15, alpha=0.82)
V_xy = np.array([affine(v) for v in V + [V[0]]])
ax.plot(V_xy[:, 0], V_xy[:, 1], color="#54a24b", lw=1.6, alpha=0.75)
for i, v in enumerate(V, start=1):
    xy = affine(v)
    ax.scatter(*xy, s=42, color="#111827", zorder=5)
    ax.text(xy[0] + 0.035, xy[1] + 0.035, f"v{i}", fontsize=8)
for line, color in [(brianchon_L14, "#e45756"), (brianchon_L25, "#f58518"), (brianchon_L36, "#7a5195")]:
    plot_line_on_box(ax, line, (-0.6, 3.0), (-1.25, 2.35), color=color, lw=1.8, alpha=0.85)
oxy = affine(brianchon_point)
ax.scatter(*oxy, s=82, color="#e45756", zorder=6)
ax.text(oxy[0] + 0.045, oxy[1] + 0.045, "O", fontsize=10, weight="bold", color="#b42318")
ax.set_xlim(-0.6, 3.0)
ax.set_ylim(-1.25, 2.35)
ax.set_title(f"Brianchon: opposite-vertex joins concurrent\nresidual {brianchon_residual:.1e}")

for ax in axes:
    ax.set_xlabel("x/z")
axes[0].set_ylabel("y/z")
fig.suptitle("Pascal and Brianchon: a theorem and its projective dual", fontsize=14, weight="bold")
pascal_path = save_fig(fig, "pascal-brianchon-duality.png")
CHECKS.update({
    "pascal_collinearity_residual": float(pascal_residual),
    "brianchon_concurrence_residual": float(brianchon_residual),
})
display_artifact(pascal_path, width=900)


In [ ]:
G = nx.DiGraph()
G.add_edges_from([
    ("conic C as RP^1", "cross-ratio on C"),
    ("cross-ratio on C", "projective bundle sweep"),
    ("line k", "H_C(k) = pair"),
    ("line l", "H_C(l) = pair"),
    ("line m", "H_C(m) = pair"),
    ("H_C(k) = pair", "quadset relation"),
    ("H_C(l) = pair", "quadset relation"),
    ("H_C(m) = pair", "quadset relation"),
    ("Matiyasevich product", "quadset relation"),
    ("quadset relation", "Pascal product cancellation"),
    ("Pascal product cancellation", "Pascal collinearity"),
    ("Pascal collinearity", "Brianchon by duality"),
])
proof_graph_is_dag = nx.is_directed_acyclic_graph(G)
pos = {
    "conic C as RP^1": (0.0, 1.0),
    "cross-ratio on C": (1.5, 1.0),
    "projective bundle sweep": (3.0, 1.0),
    "line k": (0.0, 0.25),
    "line l": (0.0, -0.25),
    "line m": (0.0, -0.75),
    "H_C(k) = pair": (1.4, 0.25),
    "H_C(l) = pair": (1.4, -0.25),
    "H_C(m) = pair": (1.4, -0.75),
    "Matiyasevich product": (2.8, -0.95),
    "quadset relation": (3.2, -0.25),
    "Pascal product cancellation": (4.9, -0.25),
    "Pascal collinearity": (6.5, -0.25),
    "Brianchon by duality": (8.1, -0.25),
}
fig, ax = plt.subplots(figsize=(12.5, 4.8))
ax.set_axis_off()
node_colors = []
for node in G.nodes:
    if node.startswith("line"):
        node_colors.append("#fef3c7")
    elif "pair" in node or "quadset" in node:
        node_colors.append("#d1fadf")
    elif "Pascal" in node or "Brianchon" in node:
        node_colors.append("#fee4e2")
    else:
        node_colors.append("#dbeafe")
nx.draw_networkx_edges(G, pos, ax=ax, arrows=True, arrowstyle="-|>", arrowsize=14, width=1.3, edge_color="#667085")
nx.draw_networkx_nodes(G, pos, ax=ax, node_color=node_colors, edgecolors="#344054", linewidths=1.0, node_size=2400)
nx.draw_networkx_labels(G, pos, ax=ax, font_size=8.3)
ax.set_title("Proof-state view: Hesse transfer supplies the one-dimensional relation used by Pascal", fontsize=13, weight="bold")
proof_graph_path = save_fig(fig, "hesse-pascal-transfer-proof-graph.png")
CHECKS["hesse_pascal_proof_graph_is_dag"] = bool(proof_graph_is_dag)
display_artifact(proof_graph_path, width=900)


## Applied Lab: Harmonic Point on a Conic

The chapter ends with a useful limit case. Fix two conic points `a` and `b`. Their tangents meet at a point `T`. For a third conic point `c`, the line `T c` meets the conic again at `d`, and the conic cross-ratio `(a,b;c,d)` is `-1`. The computation below performs the construction on the parabola and solves for the second intersection parameter.


In [ ]:
def second_parabola_intersection_parameter(line, known_t):
    coeff = [line[1], line[0], line[2]]
    roots = np.roots(coeff)
    roots = np.real_if_close(roots, tol=1000)
    other = [float(np.real(r)) for r in roots if abs(float(np.real(r)) - known_t) > 1e-7]
    if not other:
        return float(np.real(roots[0]))
    return other[0]

harm_a, harm_b, harm_c = -1.35, 1.05, 0.28
tangent_a = parabola_tangent(harm_a)
tangent_b = parabola_tangent(harm_b)
T_ab = meet(tangent_a, tangent_b)
line_to_c = join(T_ab, parabola_point(harm_c))
harm_d = second_parabola_intersection_parameter(line_to_c, harm_c)
harmonic_value = cross_ratio(harm_a, harm_b, harm_c, harm_d)
harmonic_error = abs(harmonic_value + 1)

CHECKS.update({
    "harmonic_constructed_parameter_d": float(harm_d),
    "harmonic_cross_ratio_value": float(harmonic_value),
    "harmonic_cross_ratio_error_from_minus_one": float(harmonic_error),
})
{
    "a": harm_a,
    "b": harm_b,
    "c": harm_c,
    "constructed_d": harm_d,
    "cross_ratio(a,b;c,d)": harmonic_value,
    "error_from_minus_one": harmonic_error,
}


## Artifact Gallery

The notebook generated these durable artifacts under the chapter artifact subtree:

![Conic perspectivity route map](../../artifacts/chapter-10-conics-and-perspectivity/figures/conic-perspectivity-route-map.png)

![Five-point conic pencil](../../artifacts/chapter-10-conics-and-perspectivity/figures/five-point-conic-pencil.png)

![Conic cross-ratio viewpoints](../../artifacts/chapter-10-conics-and-perspectivity/figures/conic-cross-ratio-viewpoints.png)

![Hesse transfer parabola multiplication](../../artifacts/chapter-10-conics-and-perspectivity/figures/hesse-transfer-parabola-multiplication.png)

![Pascal Brianchon duality](../../artifacts/chapter-10-conics-and-perspectivity/figures/pascal-brianchon-duality.png)

![Hesse Pascal proof graph](../../artifacts/chapter-10-conics-and-perspectivity/figures/hesse-pascal-transfer-proof-graph.png)

Open the interactive [perspective generation sweep](../../artifacts/chapter-10-conics-and-perspectivity/html/perspective-generation-sweep.html) and [projective transform conic action](../../artifacts/chapter-10-conics-and-perspectivity/html/projective-transform-conic-action.html) when reading outside an executed notebook.


## Takeaways

- A conic through five general points can be found by a linear nullspace, but the bracket pencil explains why four points leave a one-parameter family.
- Four points on a conic have a cross-ratio independent of the viewpoint on the same conic; this is the working reason a conic models `RP^1`.
- Projectively corresponding line bundles generate a conic by their intersections, while Mobius transformations of the parameter line lift to conic-preserving projective transformations.
- Hesse's transfer principle turns lines into point pairs on the conic. Concurrence becomes a quadrilateral-set relation, which is exactly the kind of one-dimensional identity used in the Pascal proof.
- Brianchon is the point-line dual of Pascal, and the same join/meet arithmetic tests both.


In [ ]:
all_artifacts = ARTIFACTS + TABLES + [CHECK_DIR / "storyboard.json"]
assert_artifacts(all_artifacts, min_size=256)

assert CHECKS["conic_design_rank"] == 5
assert CHECKS["five_point_max_residual"] < 1e-9
assert CHECKS["pencil_max_residual"] < 1e-9
assert CHECKS["svd_pencil_matrix_residual"] < 1e-9
assert CHECKS["cross_ratio_error"] < 1e-9
assert CHECKS["sweep_max_line_incidence_error"] < 1e-9
assert CHECKS["sweep_max_conic_residual"] < 1e-9
assert CHECKS["projective_transform_form_residual"] < 1e-9
assert CHECKS["projective_transform_cross_ratio_error"] < 1e-9
assert CHECKS["hesse_matiyasevich_symbolic_det_zero"]
assert CHECKS["hesse_quadset_residual"] < 1e-9
assert CHECKS["hesse_common_intercept_error"] < 1e-9
assert CHECKS["pascal_collinearity_residual"] < 1e-9
assert CHECKS["brianchon_concurrence_residual"] < 1e-9
assert CHECKS["hesse_pascal_proof_graph_is_dag"]
assert CHECKS["harmonic_cross_ratio_error_from_minus_one"] < 1e-9

artifact_stats = []
for path in ARTIFACTS:
    if path.suffix.lower() == ".png":
        stat = image_stats(path)
        stat["path"] = book_relative(stat["path"])
        artifact_stats.append(stat)
    else:
        artifact_stats.append({"path": book_relative(path), "file_size": int(path.stat().st_size)})

visual_checks = {
    "chapter": 10,
    "all_files_exist": all(path.exists() for path in all_artifacts),
    "cross_ratio_error": CHECKS["cross_ratio_error"],
    "numeric_checks": CHECKS,
    "artifacts": [book_relative(path) for path in ARTIFACTS],
    "tables": [book_relative(path) for path in TABLES],
    "artifact_stats": artifact_stats,
}
save_json(visual_checks, ARTIFACT_ROOT, "checks", "visual-checks.json")

final = {
    "chapter": 10,
    "source_span": "printed pp. 167-188 / PDF pp. 189-210",
    "notebook_executed": True,
    "artifacts": [book_relative(path) for path in ARTIFACTS],
    "tables": [book_relative(path) for path in TABLES],
    "checks": CHECKS,
}
save_json(final, ARTIFACT_ROOT, "checks", "final-sanity.json")
final
